
# Generate Pseudo-Sentence Word-Level Contextual Embeddings

This notebook creates **word-level contextual embeddings** from the pseudo-sentences built from Eyal's filtered words.

Goal:

```text
Eyal word-level file: 1735 timestamped word events
        ↓
pseudo-sentences built from Eyal's words
        ↓
XLM-RoBERTa over each pseudo-sentence
        ↓
one 768-dimensional contextual vector for each original word event
```

The output is one embedding matrix per language:

```text
English: (1735, 768)
Hebrew:  (1735, 768)
Arabic:  (1735, 768)
```


In [ ]:

# =========================
# Cell 1: Imports and paths
# =========================

import os
import re
import ast
import json
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# -------------------------
# Project paths
# -------------------------
PROJECT_ROOT = r"C:\Users\mayat\OneDrive\Desktop\lab\Language-Project-main"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
OUTPUT_DIR = os.path.join(PROCESSED_DIR, "PseudoSentencesEmbeddings")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Eyal's word-level file: must contain start, end, en, he, ar
WORD_LEVEL_PATH = os.path.join(
    DATA_DIR,
    "Amirim_Project_Submission",
    "Amirim_Project_Submission",
    "translated_podcast_transcript_filtered.csv"
)

# If your filtered transcript is somewhere else, uncomment and edit this:
# WORD_LEVEL_PATH = os.path.join(DATA_DIR, "translated_podcast_transcript_filtered.csv")

# Pseudo-sentence files created earlier
EN_PSEUDO_PATH = os.path.join(
    PROCESSED_DIR,
    "eyal_filtered_words_pseudo_sentences_en_real_sentence_boundaries_ALL_WORDS_PUNCT_IGNORED.csv"
)

HE_PSEUDO_PATH = os.path.join(
    PROCESSED_DIR,
    "eyal_filtered_words_pseudo_sentences_he_real_sentence_boundaries_ALL_WORDS_PUNCT_IGNORED.csv"
)

AR_PSEUDO_PATH = os.path.join(
    PROCESSED_DIR,
    "eyal_filtered_words_pseudo_sentences_ar_real_sentence_boundaries_ALL_WORDS_PUNCT_IGNORED_WITH_EN.csv"
)

print("WORD_LEVEL_PATH:", WORD_LEVEL_PATH)
print("EN_PSEUDO_PATH:", EN_PSEUDO_PATH)
print("HE_PSEUDO_PATH:", HE_PSEUDO_PATH)
print("AR_PSEUDO_PATH:", AR_PSEUDO_PATH)


In [ ]:

# =========================
# Cell 2: Helper functions
# =========================

def robust_read_csv(path):
    """
    Read CSV with utf-8-sig first, then fallback to utf-8.
    """
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="utf-8")


def parse_word_indices(value):
    """
    Parse a word_indices cell.
    Supports formats like:
      "[1, 2, 3]"
      "1,2,3"
      "1 2 3"
      "1|2|3"
    """
    if isinstance(value, (list, tuple, np.ndarray)):
        return [int(x) for x in value]

    if pd.isna(value):
        return []

    s = str(value).strip()
    if not s:
        return []

    # Try Python literal list first
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return [int(x) for x in parsed]
    except Exception:
        pass

    # Fallback: extract all integers
    nums = re.findall(r"\d+", s)
    return [int(x) for x in nums]


def clean_token_for_display(token):
    """
    Minimal cleanup for pseudo-sentence tokens.
    This is only for splitting/display, not linguistic normalization.
    """
    token = str(token).strip()
    token = token.strip(' .,!?"\'()[]{}:;،؟؛')
    return token


def split_text_to_tokens(text):
    """
    Split pseudo-sentence text into whitespace tokens.
    """
    text = str(text).strip()
    if not text or text.lower() == "nan":
        return []
    return [clean_token_for_display(tok) for tok in text.split() if clean_token_for_display(tok)]


def target_to_components(value):
    """
    Convert one of Eyal's target words/phrases into components.

    Examples:
      few_years -> [few, years]
      copyright_law -> [copyright, law]
      الاستخدام العادل -> [الاستخدام, العادل]
    """
    s = str(value).strip()
    s = s.replace("_", " ")
    return split_text_to_tokens(s)


def average_vectors(vectors):
    """
    Average a list/array of vectors into one vector.
    """
    vectors = np.asarray(vectors)
    if vectors.ndim == 1:
        return vectors
    return vectors.mean(axis=0)


def vector_to_string(vec):
    """
    Store vector compactly in CSV as a space-separated string inside brackets.
    """
    return "[" + " ".join(f"{x:.8g}" for x in vec) + "]"


In [ ]:

# =========================
# Cell 3: Load data
# =========================

word_level_df = robust_read_csv(WORD_LEVEL_PATH)
en_pseudo_df = robust_read_csv(EN_PSEUDO_PATH)
he_pseudo_df = robust_read_csv(HE_PSEUDO_PATH)
ar_pseudo_df = robust_read_csv(AR_PSEUDO_PATH)

print("word_level_df:", word_level_df.shape)
print("en_pseudo_df:", en_pseudo_df.shape)
print("he_pseudo_df:", he_pseudo_df.shape)
print("ar_pseudo_df:", ar_pseudo_df.shape)

print("\nword_level_df columns:")
print(word_level_df.columns.tolist())

print("\nEN pseudo columns:")
print(en_pseudo_df.columns.tolist())

print("\nHE pseudo columns:")
print(he_pseudo_df.columns.tolist())

print("\nAR pseudo columns:")
print(ar_pseudo_df.columns.tolist())

required_word_cols = {"start", "end", "en", "he", "ar"}
missing = required_word_cols - set(word_level_df.columns)
assert not missing, f"Missing columns in word-level file: {missing}"

assert len(word_level_df) == 1735, f"Expected 1735 word events, got {len(word_level_df)}"

display(word_level_df.head())
display(en_pseudo_df.head())


In [ ]:

# =========================
# Cell 4: Language specifications
# =========================

LANGUAGE_SPECS = {
    "en": {
        "word_col": "en",
        "pseudo_df": en_pseudo_df,
        "pseudo_sentence_col": "pseudo_sentence_en_from_all_eyal_words_punct_ignored",
        "output_prefix": "en_pseudo_sentence_word_context",
    },
    "he": {
        "word_col": "he",
        "pseudo_df": he_pseudo_df,
        "pseudo_sentence_col": "pseudo_sentence_he_from_all_eyal_words_punct_ignored",
        "output_prefix": "he_pseudo_sentence_word_context",
    },
    "ar": {
        "word_col": "ar",
        "pseudo_df": ar_pseudo_df,
        "pseudo_sentence_col": "pseudo_sentence_ar_from_all_eyal_words_punct_ignored",
        "output_prefix": "ar_pseudo_sentence_word_context",
    },
}

# Check that all expected columns exist
for lang, spec in LANGUAGE_SPECS.items():
    pseudo_df = spec["pseudo_df"]
    col = spec["pseudo_sentence_col"]
    assert col in pseudo_df.columns, f"Missing pseudo sentence column for {lang}: {col}"
    assert "word_indices" in pseudo_df.columns, f"Missing word_indices column for {lang}"

print("Language specs are ready.")


In [ ]:

# =========================
# Cell 5: Load XLM-RoBERTa
# =========================

MODEL_NAME = "xlm-roberta-base"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# AutoTokenizer should load the fast tokenizer, which supports word_ids()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

EMBED_DIM = model.config.hidden_size
print("Embedding dimension:", EMBED_DIM)


In [ ]:

# =========================
# Cell 6: XLM-R tokenization helpers
# =========================

@torch.no_grad()
def get_contextual_word_vectors_from_tokens(tokens):
    """
    Run XLM-RoBERTa on a list of pre-split words and return one vector per word.

    XLM-R returns vectors for subword tokens. We average all subword vectors
    belonging to the same original word.

    Returns:
      word_vectors: np.ndarray, shape (len(tokens), 768)
    """
    if not tokens:
        return np.zeros((0, EMBED_DIM), dtype=np.float32)

    encoded = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=512,
    )

    word_ids = encoded.word_ids(batch_index=0)
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    outputs = model(**encoded)
    hidden = outputs.last_hidden_state[0].detach().cpu().numpy()  # tokens x 768

    # Collect subword vectors by original word index
    vectors_by_word = {i: [] for i in range(len(tokens))}

    for token_pos, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id in vectors_by_word:
            vectors_by_word[word_id].append(hidden[token_pos])

    word_vectors = []
    for i in range(len(tokens)):
        pieces = vectors_by_word.get(i, [])
        if pieces:
            word_vectors.append(np.mean(pieces, axis=0))
        else:
            # This can happen only if truncation removed the word.
            word_vectors.append(np.full(EMBED_DIM, np.nan, dtype=np.float32))

    return np.vstack(word_vectors).astype(np.float32)


@torch.no_grad()
def get_isolated_vector(text):
    """
    Fallback: run a word/phrase alone through XLM-R and average non-special token vectors.
    """
    tokens = split_text_to_tokens(str(text))
    if not tokens:
        return np.zeros(EMBED_DIM, dtype=np.float32)

    encoded = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=512,
    )
    word_ids = encoded.word_ids(batch_index=0)
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    outputs = model(**encoded)
    hidden = outputs.last_hidden_state[0].detach().cpu().numpy()

    real_token_vectors = [hidden[i] for i, wid in enumerate(word_ids) if wid is not None]
    if not real_token_vectors:
        return np.zeros(EMBED_DIM, dtype=np.float32)

    return np.mean(real_token_vectors, axis=0).astype(np.float32)


In [ ]:

# =========================
# Cell 7: Generate word-level embeddings from pseudo-sentences
# =========================

def generate_embeddings_for_language(lang, spec):
    """
    Generate one contextual vector per original Eyal word event.

    Method:
      1. Read each pseudo-sentence.
      2. Run XLM-R on the whole pseudo-sentence.
      3. Use word_indices to map spans of pseudo-sentence tokens back to Eyal word rows.
      4. For multi-word targets, average the token-level word vectors over the span.
      5. If something fails, use isolated word fallback.
    """
    word_col = spec["word_col"]
    pseudo_df = spec["pseudo_df"].copy()
    pseudo_sentence_col = spec["pseudo_sentence_col"]
    output_prefix = spec["output_prefix"]

    n_words = len(word_level_df)
    embeddings = np.full((n_words, EMBED_DIM), np.nan, dtype=np.float32)
    flags = []

    seen_word_indices = set()

    print("\n" + "=" * 70)
    print(f"Generating embeddings for language: {lang}")
    print("=" * 70)

    for row_idx, row in tqdm(pseudo_df.iterrows(), total=len(pseudo_df), desc=f"{lang} pseudo-sentences"):
        sentence_id = row.get("sentence_id", row_idx + 1)
        pseudo_sentence = row[pseudo_sentence_col]
        word_indices = parse_word_indices(row.get("word_indices", ""))

        # Remove invalid indices defensively
        word_indices = [wi for wi in word_indices if 0 <= wi < n_words]

        if not word_indices:
            continue

        pseudo_tokens = split_text_to_tokens(pseudo_sentence)

        # Run XLM-R on the full pseudo-sentence once
        if pseudo_tokens:
            pseudo_word_vectors = get_contextual_word_vectors_from_tokens(pseudo_tokens)
        else:
            pseudo_word_vectors = np.zeros((0, EMBED_DIM), dtype=np.float32)

        token_pointer = 0

        for wi in word_indices:
            target_text = word_level_df.loc[wi, word_col]
            components = target_to_components(target_text)
            span_len = max(1, len(components))

            span_start = token_pointer
            span_end = token_pointer + span_len

            if span_end <= len(pseudo_word_vectors):
                span_vectors = pseudo_word_vectors[span_start:span_end]

                # If any vector is NaN due to truncation, fallback
                if np.isnan(span_vectors).any():
                    vec = get_isolated_vector(target_text)
                    quality_flag = "isolated_word_fallback_due_to_truncation"
                else:
                    vec = average_vectors(span_vectors)
                    quality_flag = "pseudo_sentence_word_context"

                embeddings[wi] = vec
                seen_word_indices.add(wi)

                flags.append({
                    "word_idx": wi,
                    "sentence_id": sentence_id,
                    "language": lang,
                    "target_text": target_text,
                    "components": " | ".join(components),
                    "span_start": span_start,
                    "span_end": span_end,
                    "pseudo_sentence": pseudo_sentence,
                    "quality_flag": quality_flag,
                })
            else:
                # If the pseudo-sentence tokens are shorter than expected, do not drop.
                vec = get_isolated_vector(target_text)
                embeddings[wi] = vec
                seen_word_indices.add(wi)

                flags.append({
                    "word_idx": wi,
                    "sentence_id": sentence_id,
                    "language": lang,
                    "target_text": target_text,
                    "components": " | ".join(components),
                    "span_start": span_start,
                    "span_end": span_end,
                    "pseudo_sentence": pseudo_sentence,
                    "quality_flag": "isolated_word_fallback_span_out_of_range",
                })

            token_pointer = span_end

    # Fill any missing word events defensively
    missing = [wi for wi in range(n_words) if wi not in seen_word_indices or np.isnan(embeddings[wi]).any()]

    for wi in missing:
        target_text = word_level_df.loc[wi, word_col]
        embeddings[wi] = get_isolated_vector(target_text)
        flags.append({
            "word_idx": wi,
            "sentence_id": None,
            "language": lang,
            "target_text": target_text,
            "components": " | ".join(target_to_components(target_text)),
            "span_start": None,
            "span_end": None,
            "pseudo_sentence": None,
            "quality_flag": "isolated_word_fallback_missing_from_pseudo_sentences",
        })

    # Final checks
    assert embeddings.shape == (n_words, EMBED_DIM)
    assert not np.isnan(embeddings).any(), f"NaN values found in {lang} embeddings"

    flags_df = pd.DataFrame(flags).sort_values("word_idx").reset_index(drop=True)

    print(f"\n{lang.upper()} report")
    print("Total word events:", n_words)
    print("Embedding shape:", embeddings.shape)
    print("Quality flags:")
    print(flags_df["quality_flag"].value_counts())

    # Save .npy matrix
    npy_path = os.path.join(OUTPUT_DIR, f"{output_prefix}_embeddings.npy")
    np.save(npy_path, embeddings)

    # Save quality flags
    flags_path = os.path.join(OUTPUT_DIR, f"{output_prefix}_quality_flags.csv")
    flags_df.to_csv(flags_path, index=False, encoding="utf-8-sig")

    # Save CSV with metadata and one embedding column as string
    out_df = word_level_df[["start", "end", "en", "he", "ar"]].copy()
    out_df.insert(0, "word_idx", np.arange(n_words))
    out_df["embedding"] = [vector_to_string(vec) for vec in embeddings]

    csv_path = os.path.join(OUTPUT_DIR, f"{output_prefix}_embeddings.csv")
    out_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print("Saved:")
    print(" ", npy_path)
    print(" ", csv_path)
    print(" ", flags_path)

    return embeddings, flags_df


In [ ]:

# =========================
# Cell 8: Run all languages
# =========================

all_embeddings = {}
all_flags = {}

for lang, spec in LANGUAGE_SPECS.items():
    embeddings, flags_df = generate_embeddings_for_language(lang, spec)
    all_embeddings[lang] = embeddings
    all_flags[lang] = flags_df

print("\nDone generating pseudo-sentence word-level contextual embeddings.")
for lang, emb in all_embeddings.items():
    print(lang, emb.shape)


In [ ]:

# =========================
# Cell 9: Summary table
# =========================

summary_rows = []

for lang, flags_df in all_flags.items():
    counts = flags_df["quality_flag"].value_counts().to_dict()
    total = len(word_level_df)

    summary_rows.append({
        "language": lang,
        "total_words": total,
        "pseudo_sentence_word_context": counts.get("pseudo_sentence_word_context", 0),
        "isolated_fallback_total": sum(v for k, v in counts.items() if k.startswith("isolated_word_fallback")),
        "context_rate": counts.get("pseudo_sentence_word_context", 0) / total,
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_DIR, "pseudo_sentence_word_context_embeddings_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary_df)
print("Saved summary to:", summary_path)



## Next step

After this notebook finishes, use the generated `.npy` or `.csv` files as the contextual feature matrices for encoding.

Expected output files:

```text
data/processed/PseudoSentencesEmbeddings/en_pseudo_sentence_word_context_embeddings.npy
data/processed/PseudoSentencesEmbeddings/he_pseudo_sentence_word_context_embeddings.npy
data/processed/PseudoSentencesEmbeddings/ar_pseudo_sentence_word_context_embeddings.npy
```

Each should have shape:

```text
(1735, 768)
```

These are word-aligned to Eyal's original file, so they can be used in the contextual encoding notebook.
